# Task 6 — Storage Strategy Benchmark

Benchmarks two partitioning strategies for the `tripdata` silver table: ingestion (write) time, storage size, file count, and query latency for the three required queries.

In [6]:
import os
import sys

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
spark = get_spark_session()
print("Project root:", PROJECT_ROOT)

Project root: /home/bogdan/repos/ID2221/LAB


## 1. Imports, paths, and the two strategies

In [7]:
import shutil
import time
from pyspark.sql import functions as F
from deltalake import DeltaTable

SOURCE_SILVER = os.path.join(PROJECT_ROOT, "data", "delta", "silver", "tripdata")
ZONE_LOOKUP = os.path.join(PROJECT_ROOT, "data", "delta", "silver", "taxi_zone_lookup")

STRATEGY_A_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "benchmark", "tripdata_by_month")
STRATEGY_B_PATH = os.path.join(PROJECT_ROOT, "data", "delta", "benchmark", "tripdata_by_day")

STRATEGIES = {
    "A: partitioned by (pickup_year, pickup_month)": (STRATEGY_A_PATH, ["pickup_year", "pickup_month"]),
    "B: partitioned by (event_date)": (STRATEGY_B_PATH, ["event_date"]),
}

## 2. Helper functions

In [8]:
def _read_delta(spark, path):
    """
    Read a Delta table\'s active Parquet files directly with Spark.
    basePath must be set explicitly: tripdata (and any partitioned
    benchmark output) is physically Hive-partitioned on disk
    (e.g. pickup_year=2024/pickup_month=1/...), and partition columns
    live only in the directory path, not inside each file\'s own schema.
    Without basePath, Spark cannot reliably reconstruct those columns.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"No data at {path} - run ingestion first")
    files = DeltaTable(path).file_uris()
    return spark.read.option("basePath", path).parquet(*files)


def _dir_stats(path: str):
    """Total size in bytes and count of .parquet data files under `path`, excluding _delta_log."""
    total_size = 0
    file_count = 0
    for root, dirs, files in os.walk(path):
        if "_delta_log" in root:
            continue
        for f in files:
            if f.endswith(".parquet"):
                full = os.path.join(root, f)
                total_size += os.path.getsize(full)
                file_count += 1
    return total_size, file_count


def _time_it(fn):
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    return result, elapsed

## 3. The 3 required benchmark queries

In [9]:
def query_trips_per_borough(df, zone_lookup):
    pickup_zone = zone_lookup.select(
        F.col("location_id").alias("pickup_location_id"),
        F.col("borough").alias("pickup_borough"),
    )
    return (
        df.join(F.broadcast(pickup_zone), on="pickup_location_id", how="left")
        .groupBy("pickup_borough").count()
        .collect()
    )


def query_avg_duration_per_day(df, zone_lookup):
    return (
        df.groupBy("event_date")
        .agg(F.avg("trip_duration_minutes").alias("avg_duration_minutes"))
        .collect()
    )


def query_avg_fare_per_borough(df, zone_lookup):
    pickup_zone = zone_lookup.select(
        F.col("location_id").alias("pickup_location_id"),
        F.col("borough").alias("pickup_borough"),
    )
    return (
        df.join(F.broadcast(pickup_zone), on="pickup_location_id", how="left")
        .groupBy("pickup_borough").agg(F.avg("fare_amount").alias("avg_fare"))
        .collect()
    )


QUERIES = {
    "Q1: number of taxi trips per borough": query_trips_per_borough,
    "Q2: average trip duration per day": query_avg_duration_per_day,
    "Q3: average fare per borough": query_avg_fare_per_borough,
}

## 4. Run the benchmark

Writes both strategies fresh (overwrite), measures write time, storage size, file count, and query latency for each.

In [11]:
source_df = _read_delta(spark, SOURCE_SILVER).filter(F.col("is_valid"))
zone_lookup = _read_delta(spark, ZONE_LOOKUP)

source_row_count = source_df.count()
print(f"Source tripdata rows (is_valid=true): {source_row_count}\n")

results = {}

for label, (path, partition_cols) in STRATEGIES.items():
    print(f"{'=' * 70}\n{label}\n{'=' * 70}")

    if os.path.exists(path):
        shutil.rmtree(path)

    # --- Ingestion time (write step only) ---
    # Only the write step is timed: load/transform/validation already
    # happened once to build the shared source_df, and that cost is
    # identical regardless of partitioning scheme - only the write step
    # is actually a function of the storage strategy being compared.
    from src.common.delta_writer import write_delta
    written, write_seconds = _time_it(
        lambda: write_delta(source_df, path, mode="overwrite", partition_by=partition_cols)
    )
    print(f"  write time:     {write_seconds:.2f}s  ({written} rows written)")

    # --- Storage size and file count ---
    size_bytes, file_count = _dir_stats(path)
    size_mb = size_bytes / (1024 * 1024)
    print(f"  storage size:   {size_mb:.2f} MB")
    print(f"  files:          {file_count}")

    # --- Query latency, re-reading fresh each time ---
    spark.catalog.clearCache()
    strategy_df = _read_delta(spark, path).filter(F.col("is_valid"))

    query_times = {}
    for q_label, q_fn in QUERIES.items():
        spark.catalog.clearCache()
        _, elapsed = _time_it(lambda: q_fn(strategy_df, zone_lookup))
        query_times[q_label] = elapsed
        print(f"  {q_label}: {elapsed:.3f}s")

    results[label] = {
        "write_seconds": write_seconds,
        "size_mb": size_mb,
        "file_count": file_count,
        "query_times": query_times,
    }
    print()

Source tripdata rows (is_valid=true): 9554776

A: partitioned by (pickup_year, pickup_month)


26/09/13 13:53:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  write time:     7.76s  (9554776 rows written)
  storage size:   891.91 MB
  files:          13
  Q1: number of taxi trips per borough: 0.738s
  Q2: average trip duration per day: 0.315s
  Q3: average fare per borough: 0.309s

B: partitioned by (event_date)


  write time:     5.39s  (9554776 rows written)
  storage size:   844.45 MB
  files:          95
  Q1: number of taxi trips per borough: 0.334s
  Q2: average trip duration per day: 0.423s
  Q3: average fare per borough: 0.315s



## 5. Build and save the results report

In [17]:
import os

def render_report(results: dict) -> str:
    lines = [
        "# Task 6 — Storage Strategy Benchmark",
        "",
        "Two partitioning strategies for the `tripdata` silver table, benchmarked "
        "on ingestion (write) time, storage size, file count, and query latency "
        "for the three required queries.",
        "",
        "| Metric | " + " | ".join(results.keys()) + " |",
        "|---|" + "---|" * len(results),
        "| Write time (s) | " + " | ".join(f"{r['write_seconds']:.2f}" for r in results.values()) + " |",
        "| Storage size (MB) | " + " | ".join(f"{r['size_mb']:.2f}" for r in results.values()) + " |",
        "| Number of files | " + " | ".join(str(r["file_count"]) for r in results.values()) + " |",
    ]
    query_labels = list(next(iter(results.values()))["query_times"].keys())
    for q in query_labels:
        row = [f"| {q} (s)"] + [f"{r['query_times'][q]:.3f}" for r in results.values()]
        lines.append(" | ".join(row) + " |")
    lines.append("")
    return "\n".join(lines)


report_text = render_report(results)

out_dir = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "benchmark_results.md")
with open(out_path, "w") as f:
    f.write(report_text)

print(f"Saved: {out_path}")

Saved: /home/bogdan/repos/ID2221/LAB/reports/benchmark_results.md


## 6. Render the results table inline

In [18]:
from IPython.display import Markdown, display
display(Markdown(report_text))

# Task 6 — Storage Strategy Benchmark

Two partitioning strategies for the `tripdata` silver table, benchmarked on ingestion (write) time, storage size, file count, and query latency for the three required queries.

| Metric | A: partitioned by (pickup_year, pickup_month) | B: partitioned by (event_date) |
|---|---|---|
| Write time (s) | 7.76 | 5.39 |
| Storage size (MB) | 891.91 | 844.45 |
| Number of files | 13 | 95 |
| Q1: number of taxi trips per borough (s) | 0.738 | 0.334 |
| Q2: average trip duration per day (s) | 0.315 | 0.423 |
| Q3: average fare per borough (s) | 0.309 | 0.315 |
